<a href="https://colab.research.google.com/github/TharinduniHerath/Deep_learning_assignment/blob/BI-LSTM/BI_LSTM_it22322708.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/DL_assignment/IMDB Dataset.csv')

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
### Drop NaN values
df = df.dropna()

In [6]:
### Get the independent Features
x = df.drop('sentiment', axis=1)

In [7]:
### Get the Dependent Feaatures
y=df['sentiment'].map({'positive':1, 'negative':0})

In [8]:
x.shape

(50000, 1)

In [9]:
y.shape

(50000,)

In [10]:
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,0


In [11]:
import tensorflow as tf

In [12]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Bidirectional

In [13]:
### Vocabulary size
voc_size = 5000

**One-Hot Representation**

In [14]:
messages = x.copy()

In [15]:
messages['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [16]:
messages.reset_index(inplace=True)

In [17]:
import re
import nltk
from nltk.corpus import stopwords
from tqdm import tqdm

In [18]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [19]:
### Data Preprocessing
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
corpus = []
for i in tqdm(range(len(messages))):
  text = re.sub('<.*?>',  ' ', messages['review'][i])
  review = re.sub('[^a-zA-Z]', ' ', text)
  review = review.lower()
  review = review.split()
  review = [ps.stem(word) for word in review if word not in stop_words]
  review = ' '.join(review)
  corpus.append(review)

100%|██████████| 50000/50000 [01:54<00:00, 438.34it/s]


In [20]:
onehot_repr = [one_hot(words, voc_size)for words in corpus]

In [21]:
sent_length = 400
embedded_docs = pad_sequences(onehot_repr, padding='pre', maxlen=sent_length)
print(embedded_docs)

[[   0    0    0 ... 1740 2860 3130]
 [   0    0    0 ... 1974 3772 2020]
 [   0    0    0 ... 2087 3629 3188]
 ...
 [   0    0    0 ... 1040  216 2630]
 [   0    0    0 ... 3484  866 4753]
 [   0    0    0 ... 1297 1324 2903]]


In [22]:
embedding_vector_features = 100

model = Sequential()
model.add(Embedding(input_dim=voc_size, output_dim=embedding_vector_features, input_length=sent_length))
model.add(Bidirectional(LSTM(100, dropout=0.2, recurrent_dropout=0.2)))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [23]:
import numpy as np
x_final = np.array(embedded_docs)
y_final = np.array(y)

In [24]:
x_final.shape, y_final.shape

((50000, 400), (50000,))

In [25]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_final, y_final, test_size=0.33, random_state=64)

In [26]:
model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=3, batch_size=64)

Epoch 1/3
524/524 ━━━━━━━━━━━━━━━━━━━━ 1300s 2s/step - accuracy: 0.7408 - loss: 0.5126 - val_accuracy: 0.8520 - val_loss: 0.3511
Epoch 2/3
524/524 ━━━━━━━━━━━━━━━━━━━━ 1325s 2s/step - accuracy: 0.8633 - loss: 0.3326 - val_accuracy: 0.8565 - val_loss: 0.3463
Epoch 3/3
524/524 ━━━━━━━━━━━━━━━━━━━━ 1270s 2s/step - accuracy: 0.8824 - loss: 0.2963 - val_accuracy: 0.8504 - val_loss: 0.3518


In [27]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
scores = model.evaluate(x_test, y_test, verbose=1, batch_size=256)
print("accuracy: %.2f%%" % (scores[1]*100))

65/65 ━━━━━━━━━━━━━━━━━━━━ 30s 444ms/step - accuracy: 0.8487 - loss: 0.3547
accuracy: 85.04%


In [28]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 400, 100)       │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 200)            │       160,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           201 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 661,001 (2.52 MB)

 Trainable params: 661,001 (2.52 MB)

 Non-trainable params: 0 (0.00 B)